# 01 · Smoke test

Goal (due 19 Oct): prove the training path runs end to end before committing to a full run.

This notebook is a thin runner. Every piece of logic lives in `src/argubot` and is tested; the
notebook only wires it together. It:

1. loads `Qwen/Qwen3.5-0.8B-Base`, falling back to `gpt2-medium` if it will not load,
2. confirms `target_modules="all-linear"` reaches the attention (and Gated DeltaNet) projections,
3. builds the LoRA adapter and prints the trainable-parameter count for the report,
4. runs a few LoRA steps on toy data on a T4 to catch fp16 NaNs and time a step,
5. generates one reply so the full prompt → generate path is exercised.

Numbers here (loss, the reply text) are meaningless — the adapter is untrained. What matters is
that nothing errors, the loss stays finite, and the parameter count and seconds/step get recorded.

## Setup

Runs anywhere. On Colab it clones the repo; on the SCIS cluster or locally it just finds the repo
root. It installs `torch`/`transformers`/`peft`/`accelerate` and the `argubot` package only if they
are missing (the cluster's PyTorch notebook ships Jupyter + CUDA but not torch, so the first run
installs them; Colab already has torch and skips).

In [ ]:
import importlib.util, os, subprocess, sys
from pathlib import Path

# Colab check that does NOT raise when the `google` package is absent (it does on the
# SCIS cluster / a local venv). find_spec on a dotted name raises if the parent is missing.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False

# On Colab, clone the repo once. Elsewhere (cluster / local) it is already present.
if IN_COLAB:
    REPO_URL = "https://github.com/diyadj/argubot.git"
    if not os.path.isdir("argubot"):
        subprocess.run(["git", "clone", REPO_URL], check=True)
    os.chdir("argubot")

# Work from the repo root so configs/config.yaml and data/toy/... resolve, no matter which
# directory the kernel started in (e.g. notebooks/ on the cluster).
for d in [Path.cwd(), *Path.cwd().parents]:
    if (d / "pyproject.toml").exists():
        os.chdir(d)
        break

# Make `argubot` importable in THIS kernel by putting src/ on the path. A `pip install -e .`
# adds a .pth that a running kernel only reads at startup, so this avoids a restart.
src = str(Path.cwd() / "src")
if src not in sys.path:
    sys.path.insert(0, src)

# Install only what is missing. Regular installs are importable immediately; the cluster's
# PyTorch notebook ships Jupyter + CUDA but not torch. numpy/pyyaml are argubot's own deps.
pip_name = {"torch": "torch", "transformers": "transformers", "peft": "peft",
            "accelerate": "accelerate", "numpy": "numpy", "yaml": "pyyaml"}
need = [pip_name[m] for m in pip_name if importlib.util.find_spec(m) is None]
if need:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *need], check=True)

# Drop any already-imported argubot modules so a "Run All" after a git pull picks up the
# updated code WITHOUT needing a kernel restart.
for _m in [k for k in list(sys.modules) if k == "argubot" or k.startswith("argubot.")]:
    del sys.modules[_m]

import argubot  # noqa: F401
print("cwd:", os.getcwd(), "| IN_COLAB:", IN_COLAB, "| argubot:", argubot.__file__)

## Config and seed

Nothing about the experiment is hardcoded here — it all comes from `configs/config.yaml`.

In [ ]:
import torch
from transformers import set_seed
from argubot.config import load_config

cfg = load_config("configs/config.yaml")
set_seed(cfg["seed"])
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device       :", device)
if device == "cuda":
    print("gpu          :", torch.cuda.get_device_name(0))
print("base_model   :", cfg["generator"]["base_model"])
print("fallback     :", cfg["generator"]["fallback_model"])

## Load the model (with fallback)

If `Qwen/Qwen3.5-0.8B-Base` will not load with Colab's transformers version, this falls back to
`gpt2-medium`. If that happens, **tell the group the same day** — the fallback changes the LoRA
target modules and the context limit. Recent transformers releases renamed `torch_dtype` to
`dtype`, so both are tried.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

USE_FP16 = bool(cfg["training"]["fp16"]) and device == "cuda"  # a T4 has no bf16

def load_base(model_name):
    tok = AutoTokenizer.from_pretrained(model_name)
    if tok.pad_token is None:            # only set pad if the tokenizer has none
        tok.pad_token = tok.eos_token
    tok.truncation_side = "left"         # keep the [response] tag; drop the oldest text instead
    dtype = torch.float16 if USE_FP16 else torch.float32
    try:
        model = AutoModelForCausalLM.from_pretrained(model_name, dtype=dtype)
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=dtype)
    return tok, model.to(device)

model_name = cfg["generator"]["base_model"]
try:
    tokenizer, base_model = load_base(model_name)
except Exception as e:
    print(f"Could not load {model_name!r}: {type(e).__name__}: {e}")
    model_name = cfg["generator"]["fallback_model"]
    print(f"Falling back to {model_name!r} -- tell the group today.")
    tokenizer, base_model = load_base(model_name)

model_type = getattr(base_model.config, "model_type", "?")
print(f"loaded: {model_name}  (model_type={model_type}, dtype={next(base_model.parameters()).dtype})")

## Confirm `all-linear` reaches the right layers

List every `nn.Linear` leaf. `target_modules="all-linear"` expands to all of them, so it covers the
attention projections (`q_proj`, `k_proj`, `v_proj`, `o_proj`) and, on Qwen3.5, the Gated DeltaNet
projections too.

In [ ]:
import torch.nn as nn
from collections import Counter

linear_names = Counter(
    name.split(".")[-1] for name, m in base_model.named_modules() if isinstance(m, nn.Linear)
)
print("nn.Linear leaf names (name x count):")
for name, c in sorted(linear_names.items(), key=lambda x: -x[1]):
    print(f"  {name:<18} x{c}")
print(f"\ntotal nn.Linear modules: {sum(linear_names.values())}")
print("config target_modules  :", cfg["lora"]["target_modules"])
print("\n'all-linear' expands to every name above, so attention and (on Qwen3.5) the")
print("Gated DeltaNet projections are all adapted.")

## Build the LoRA adapter

`build_lora_config` reads r / alpha / target_modules from the config. Record the trainable count
and percentage — the report asks for both (it should be roughly 1%).

In [ ]:
from peft import get_peft_model
from argubot.generate.training import build_lora_config

# all-linear works for Qwen3.5; GPT-2 names its attention projection c_attn instead.
if model_type == "gpt2":
    cfg["lora"]["target_modules"] = ["c_attn"]
    print("GPT-2 fallback: overriding target_modules to ['c_attn'].")

model = get_peft_model(base_model, build_lora_config(cfg))

# fp16 + PEFT raises "Attempting to unscale FP16 gradients"; keep the trainable params in fp32.
if USE_FP16:
    for p in model.parameters():
        if p.requires_grad:
            p.data = p.data.float()

model.print_trainable_parameters()
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"\nRECORD FOR REPORT -> trainable: {trainable:,} / {total:,} = {100 * trainable / total:.3f}%")

## One LoRA step on toy data

Uses the real `ReplyOnlyDataset` and `ReplyOnlyCollator` on `data/toy/toy_train_examples.jsonl`,
so the exact training path (reply-only labels, per-batch padding, slot blanking) is exercised. A
few steps are enough to catch fp16 NaNs and time a step. No `GradScaler`, so no unscale error.

In [ ]:
import time
from torch.utils.data import DataLoader
from argubot.io import read_jsonl
from argubot.generate.training import ReplyOnlyDataset, ReplyOnlyCollator

examples = list(read_jsonl("data/toy/toy_train_examples.jsonl"))
dataset = ReplyOnlyDataset(examples, tokenizer, cfg, train=True, seed=cfg["seed"])
collate = ReplyOnlyCollator(tokenizer.pad_token_id)
batch_size = min(cfg["training"]["per_device_batch_size"], len(dataset))
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, collate_fn=collate)

optim = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=cfg["training"]["learning_rate"])
model.train()

N_STEPS = 3
step_times = []
data_iter = iter(loader)
for step in range(N_STEPS):
    try:
        batch = next(data_iter)
    except StopIteration:
        data_iter = iter(loader)
        batch = next(data_iter)
    batch = {k: v.to(device) for k, v in batch.items()}
    t0 = time.time()
    optim.zero_grad()
    loss = model(**batch).loss
    loss.backward()
    optim.step()
    if device == "cuda":
        torch.cuda.synchronize()
    dt = time.time() - t0
    step_times.append(dt)
    print(f"step {step + 1}/{N_STEPS}  loss={loss.item():.4f}  {dt:.2f}s")
    assert torch.isfinite(loss), (
        "Loss is NaN/Inf -- fp16 on this GPU is unstable. "
        "Set training.fp16=false (load fp32) or move to the SCIS GPU."
    )

sec_per_step = sum(step_times[1:]) / max(len(step_times) - 1, 1)  # skip step 1 (warmup/compile)
print(f"\nRECORD FOR REPORT -> seconds/step (batch={batch_size}, excl. warmup): {sec_per_step:.2f}s")

## Generate one reply

Builds the prompt with the shared `build_prompt` (same layout as training) and generates with the
decoding settings from the config. The reply is gibberish — the adapter has had three toy steps —
but it proves the prompt → generate path works.

In [ ]:
from argubot.schemas import Turn
from argubot.prompt import build_prompt

model.eval()
history = [Turn(speaker="user", text="School uniforms should be mandatory because they cut down on bullying.")]
prompt = build_prompt(
    history,
    claims=[],
    passages=["School uniforms reduce bullying because clothing differences matter less."],
    max_history_turns=cfg["prompt"]["max_history_turns"],
    max_passage_words=cfg["prompt"]["max_passage_words"],
)
print("PROMPT:\n" + prompt)

enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=cfg["training"]["max_seq_len"]).to(device)
with torch.no_grad():
    out = model.generate(
        **enc,
        do_sample=True,
        temperature=cfg["decoding"]["temperature"],
        top_p=cfg["decoding"]["top_p"],
        max_new_tokens=cfg["decoding"]["max_new_tokens"],
        pad_token_id=tokenizer.pad_token_id,
    )
reply = tokenizer.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
print("\nREPLY (untrained adapter, content is meaningless):\n" + reply.strip())

## Summary to copy into the report log

Fill in the NaN / OOM line by hand with anything that went wrong and how it was fixed.

In [ ]:
print("=== Smoke-test summary ===")
print(f"base model       : {model_name}  (model_type={model_type})")
print(f"device           : {device}" + (f" / {torch.cuda.get_device_name(0)}" if device == 'cuda' else ""))
print(f"dtype / fp16     : {next(model.parameters()).dtype}, USE_FP16={USE_FP16}")
print(f"LoRA             : r={cfg['lora']['r']}, alpha={cfg['lora']['alpha']}, target={cfg['lora']['target_modules']}")
print(f"trainable params : {trainable:,} / {total:,} = {100 * trainable / total:.3f}%")
print(f"seconds/step     : {sec_per_step:.2f}s  (batch={batch_size})")
print( "NaN / OOM notes  : <fill in: any NaNs? any OOM? how resolved?>")